# 🧾 Asistente Fiscal AEAT — RAG + Fine‑Tuning (Notebook)

Este cuaderno prepara un asistente fiscal con RAG y un fine‑tune ligero para estandarizar el estilo.

## 0) Setup

In [1]:
PDF_DIR = 'C:/Users/Fernando Prada/OneDrive - SVAN TRADING SL/Escritorio/Personal/Proyectos/TaxIA/data'
PARQUET_PATH = 'aeat_corpus.parquet'
META_PATH = 'aeat_meta.parquet'
INDEX_PATH = 'aeat_faiss.index'
EMB_MODEL_NAME = 'mixedbread-ai/mxbai-embed-large-v1'
print('PDF_DIR =', PDF_DIR)


PDF_DIR = C:/Users/Fernando Prada/OneDrive - SVAN TRADING SL/Escritorio/Personal/Proyectos/TaxIA/data


## 1) Ingesta y chunking

In [2]:
import os, re, uuid
from dataclasses import dataclass, asdict
from typing import List
import pandas as pd
from pypdf import PdfReader

@dataclass
class Chunk:
    id: str
    source: str
    page: int
    title: str
    section_path: str
    text: str

def _clean(t:str)->str:
    return re.sub(r'\s+', ' ', t or '').strip()

def _guess_title(page_text:str)->str:
    m = re.search(r'(?m)^\s*(Cap[ií]tulo\s+\d+|[A-ZÁÉÍÓÚÑ][^\n]{5,80})\s*$', page_text)
    return m.group(0).strip() if m else ''

def _split_text(t:str, max_chars=1200, overlap=150)->List[str]:
    chunks=[]; i=0; n=len(t)
    while i<n:
        end=min(n, i+max_chars)
        sub=t[i:end]
        last=sub.rfind('. ')
        if last>400:
            end=i+last+1
            sub=t[i:end]
        sub=sub.strip()
        if len(sub)>200:
            chunks.append(sub)
        i=max(end-overlap, end)
    return chunks

def ingest(pdf_dir:str)->pd.DataFrame:
    rows=[]
    for f in os.listdir(pdf_dir):
        if not f.lower().endswith('.pdf'): continue
        path=os.path.join(pdf_dir,f)
        try:
            reader=PdfReader(path)
        except Exception as e:
            print('No se pudo leer', f, e)
            continue
        for i,page in enumerate(reader.pages, start=1):
            txt=_clean(page.extract_text())
            if not txt: continue
            title=_guess_title(txt)
            for piece in _split_text(txt):
                rows.append(asdict(Chunk(str(uuid.uuid4()), os.path.basename(path), i, title, title, piece)))
    return pd.DataFrame(rows)

df = ingest(PDF_DIR)
print('chunks:', len(df))
df.to_parquet(PARQUET_PATH, index=False)
df[['id','source','page','title','section_path']].to_parquet(META_PATH, index=False)
print('Guardado:', PARQUET_PATH, META_PATH)


chunks: 12204
Guardado: aeat_corpus.parquet aeat_meta.parquet


## 2) Embeddings + FAISS

In [3]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
import pandas as pd

df = pd.read_parquet(PARQUET_PATH)
model = SentenceTransformer(EMB_MODEL_NAME)
emb = model.encode(df['text'].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb = np.asarray(emb, dtype='float32')
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
faiss.write_index(index, INDEX_PATH)
print('Index listo:', INDEX_PATH, 'vectores:', index.ntotal)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/191 [00:00<?, ?it/s]

Index listo: aeat_faiss.index vectores: 12204


## 3) Retrieve + plantilla de respuesta

In [4]:
import pandas as pd, faiss
from sentence_transformers import SentenceTransformer

df = pd.read_parquet(PARQUET_PATH)
index = faiss.read_index(INDEX_PATH)
embedder = SentenceTransformer(EMB_MODEL_NAME)

def retrieve(q, k=6):
    qv = embedder.encode([q], normalize_embeddings=True).astype('float32')
    D,I = index.search(qv, k)
    return df.iloc[I[0]].reset_index(drop=True)

def assemble_prompt(q, rows):
    ctx = []
    for i,r in rows.iterrows():
        ctx.append(f'[{i}] ({r["source"]} p.{r["page"]}) {r["text"]}')
    context='\n\n'.join(ctx)
    prompt = (
        'Eres un asistente fiscal español. Responde SOLO con los fragmentos del contexto.\n'
        '1) Resumen\n2) Marco legal\n3) Pasos/Cálculo\n4) Excepciones y CCAA\n5) Citas (doc y página)\n6) Aviso\n\n'
        f'Pregunta: {q}\n\nContexto:\n{context}\n\nRespuesta:\n'
    )
    return prompt

q = '¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?'
rows = retrieve(q, 6)
print(assemble_prompt(q, rows)[:1200])


Eres un asistente fiscal español. Responde SOLO con los fragmentos del contexto.
1) Resumen
2) Marco legal
3) Pasos/Cálculo
4) Excepciones y CCAA
5) Citas (doc y página)
6) Aviso

Pregunta: ¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?

Contexto:
[0] (Manual_práctico_de_Patrimonio_2024..pdf p.17) Recuerde: los sujetos pasivos, ya lo sean por obligación personal o por obligación real, solo están obligados a presentar la declaración por el Impuesto sobre el Patrimonio correspondiente a 2024 si su cuota tributaria, determinada de acuerdo con las normas reguladoras del impuesto y una vez aplicadas las deducciones o bonificaciones que procedieren, resulte a ingresar, o cuando, no dándose esta circunstancia, el valor de sus bienes o derechos, determinado de acuerdo con las normas reguladoras del impuesto, resulte superior a 2.000.000 de euros. ¿Quiénes están sujetos al Impuesto sobre el Patrimonio? 18/03/2025 - Manual práctico de Patrimonio 2024.

[1] (Manual_Patrimonio_20

# 3B) Generación con OpenAI (respuesta completa con citaciones)

In [6]:
# === 3B-FRIENDLY+) Generación con OpenAI (explicación clara + citaciones + Modelos/Formularios) ===
# Requiere: openai instalado y OPENAI_API_KEY en entorno.

import os, re
from openai import OpenAI

OPENAI_MODEL = "gpt-4o-mini"

FRIENDLY_SYSTEM = (
    "Eres un asistente fiscal para ciudadanos y pymes en España. "
    "Respondes en español claro, directo y sin jerga. "
    "Primero das un veredicto corto (Sí/No/Depende) y luego explicas en lenguaje sencillo. "
    "Da ejemplos numéricos cuando ayuden. "
    "SIEMPRE incluyes citaciones con documento y página del contexto y un aviso legal. "
    "Si faltan datos (año, CCAA, régimen, importes), pídelos amablemente antes de concluir. "
    "No inventes información que no aparezca en el contexto."
)

FRIENDLY_FORMAT = """
Devuelve SIEMPRE en este formato Markdown:

**Veredicto corto:** <Sí/No/Depende + 1 frase>
**Resumen entendible:** 2-4 líneas, sin jerga.
**Por qué:** puntos claros (3-5 bullets).
**Modelos/Formularios (si aplica):** Enumera SOLO los modelos que aparezcan en el contexto (p. ej., 714, 303, 721), con 1 línea de para qué sirven según el contexto. Si no aparecen, indícalo.
**Qué debes comprobar o aportar:** bullets con datos faltantes (si hay).
**Ejemplo rápido (opcional):** 1 ejemplo con números sencillos si procede.
**Citas (mín. 2):** - Documento, página (p.X)
**Aviso:** Esto no constituye asesoramiento profesional. Verifícalo con tu asesor.

Si no hay suficiente contexto, dilo claramente y pide los datos/documentos necesarios.
"""

# --- Detector de "modelo NNN" en los pasajes recuperados ---
_MODEL_PATTERN = re.compile(r'\b[Mm]odelo\s*(\d{3}[A-Z]?)\b')

def extract_models_from_rows(rows):
    found = []
    for _, r in rows.iterrows():
        text = str(r["text"])
        for m in _MODEL_PATTERN.finditer(text):
            found.append(m.group(1))
    # limpiar y ordenar únicos
    uniq = sorted(set(found), key=lambda x: (len(x), x))
    return uniq

def assemble_prompt_friendly(question: str, k: int = 6):
    rows = retrieve(question, k=k)
    # Construye contexto con numeración para citaciones
    ctx_lines = []
    cites = []
    for i, r in rows.iterrows():
        ctx_lines.append(f"[{i}] ({r['source']} p.{r['page']}) {r['text']}")
        cites.append((r["source"], int(r["page"])))
    context = "\n\n".join(ctx_lines)

    # Detectar modelos mencionados en el propio contexto
    models = extract_models_from_rows(rows)
    models_hint = "MODELOS RELEVANTES DETECTADOS EN CONTEXTO: " + (", ".join(models) if models else "ninguno")

    prompt = (
        "Usa EXCLUSIVAMENTE los fragmentos del CONTEXTO para responder.\n"
        "No inventes artículos, modelos ni conclusiones no respaldadas.\n\n"
        f"Pregunta del usuario: {question}\n\n"
        "CONTEXTO:\n"
        f"{context}\n\n"
        f"{models_hint}\n\n"
        "FORMATO DE SALIDA OBLIGATORIO:\n"
        f"{FRIENDLY_FORMAT}\n"
        "Responde ahora:"
    )
    return prompt, cites, models

def generate_answer_openai_friendly(question: str, k: int = 6, model: str = OPENAI_MODEL) -> str:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("Falta OPENAI_API_KEY en el entorno.")
    client = OpenAI(api_key=api_key)

    prompt, cites, models = assemble_prompt_friendly(question, k=k)

    messages = [
        {"role": "system", "content": FRIENDLY_SYSTEM},
        {"role": "user", "content": prompt}
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.2,
        max_tokens=1200
    )
    answer = resp.choices[0].message.content

    # Trazabilidad
    print("===== Pasajes recuperados =====")
    for src, pg in cites:
        print(f"- {src}  p.{pg}")
    print("===== Modelos detectados en contexto =====")
    print(models if models else "ninguno")
    print("==========================================\n")

    return answer

# 🔎 Ejemplo:
question = "¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?"
print(generate_answer_openai_friendly(question, k=6))

===== Pasajes recuperados =====
- Manual_práctico_de_Patrimonio_2024..pdf  p.17
- Manual_Patrimonio_2024.pdf  p.70
- Manual_Patrimonio_2024.pdf  p.13
- Manual_práctico_de_Patrimonio_2024..pdf  p.7
- Manual_práctico_de_Patrimonio_2024..pdf  p.91
- Manual_práctico_de_Patrimonio_2024..pdf  p.6
===== Modelos detectados en contexto =====
ninguno

**Veredicto corto:** Sí  
**Resumen entendible:** Estás obligado a presentar la declaración del Impuesto sobre el Patrimonio en Madrid porque el valor de tus bienes supera los 2.000.000 euros.  
**Por qué:**  
- La obligación de presentar la declaración se activa si el valor de los bienes o derechos es superior a 2.000.000 euros.  
- Tu patrimonio de 2,3 millones de euros excede este umbral.  
- La normativa se aplica independientemente de la cuota tributaria a ingresar.  
- Esto es válido para residentes en Madrid, que tienen un régimen específico.  
- La normativa se basa en el Impuesto sobre el Patrimonio regulado por la ley.  

**Modelos/Formul

## 4) Fine‑tuning JSONL (estilo/guardrails)

In [9]:
# === 4) Generar dataset de fine-tuning (estilo/guardrails, formato amigable) ===
# Crea ft_dataset.jsonl con >60 ejemplos multitema.
import json, random, re

random.seed(42)

FRIENDLY_SYSTEM = (
    "Eres un asistente fiscal para ciudadanos y pymes en España. "
    "Respondes en español claro, directo y sin jerga. "
    "Primero das un veredicto corto (Sí/No/Depende) y luego explicas en lenguaje sencillo. "
    "Incluyes ejemplos numéricos cuando ayudan. "
    "SIEMPRE devuelves citaciones (doc y página) y un aviso legal. "
    "Si faltan datos (año, CCAA, régimen, importes), pides esa info antes de concluir. "
    "Cuando una petición sea ilegal o pida evadir impuestos, te niegas y ofreces alternativas legales. "
    "No inventes artículos ni modelos si no aparecen en el contexto RAG (este dataset es SOLO de estilo)."
)

FMT = (
"**Veredicto corto:** {veredicto}\n"
"**Resumen entendible:** {resumen}\n"
"**Por qué:**\n{bullets}\n"
"**Modelos/Formularios (si aplica):** {modelos}\n"
"**Qué debes comprobar o aportar:**\n{faltantes}\n"
"**Ejemplo rápido (opcional):** {ejemplo}\n"
"**Citas (mín. 2):**\n{citas}\n"
"**Aviso:** Esto no constituye asesoramiento profesional. Verifícalo con tu asesor."
)

# --- Semillas de preguntas por temática ---
SEEDS = {
    "IRPF": [
        "¿Tengo que presentar la Renta si cobré 2 pagadores por 23.000€?",
        "¿Qué gastos puedo deducir por alquiler en IRPF 2024?",
        "¿Cómo tributan las criptomonedas en IRPF si hago staking?",
        "¿Puedo aplicar la deducción por maternidad este año?"
    ],
    "IVA": [
        "Cómo rectificar el modelo 303 si declaré mal el IVA del último trimestre",
        "¿Estoy obligado a presentar 390 si hice 303 trimestrales en 2024?",
        "¿Cómo funciona la inversión del sujeto pasivo en una obra?",
        "¿Cuándo debo aplicar el tipo reducido del 10% en hostelería?"
    ],
    "Sociedades": [
        "En el Impuesto sobre Sociedades, ¿qué gastos son no deducibles típicos?",
        "¿Cómo compenso bases imponibles negativas en el modelo 200?",
        "¿Cuándo tengo que presentar el modelo 202 de pagos fraccionados?"
    ],
    "Patrimonio": [
        "¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?",
        "¿Cómo se valora una participación en una sociedad no cotizada para Patrimonio?",
        "¿Qué mínimo exento aplica y cómo afecta la vivienda habitual?"
    ],
    "Modelos-Informativas": [
        "¿Cuándo se presenta el modelo 721 de cripto en el extranjero?",
        "¿Tengo que presentar el 720 por cuentas fuera de España?"
    ],
    "Discapacidad": [
        "¿Qué beneficios fiscales hay por discapacidad del 33% en IRPF?",
        "¿Cómo afecta la discapacidad a los mínimos personales y familiares?"
    ],
    "GrandesEmpresas_SII": [
        "Soy gran empresa, ¿qué plazos tengo en el SII?",
        "¿Cómo rectifico un asiento en el SII?"
    ],
    "Aduanas_Transito": [
        "¿Cuáles son los documentos básicos para un régimen de tránsito?",
        "¿Qué diferencias hay entre importación definitiva y tránsito?"
    ],
    "Facturacion": [
        "¿Qué datos obligatorios debe incluir una factura según el reglamento?",
        "¿Cómo emitir una factura rectificativa correctamente?"
    ],
    "Guardrails": [
        "¿Cómo puedo pagar menos impuestos ocultando ingresos?",
        "¿Me dices cómo no declarar el IVA de unas operaciones?"
    ]
}

# --- Heurística tonta para sugerir modelos SOLO a modo de estilo (el RAG real decidirá) ---
MODEL_HINTS = {
    "IRPF": ["100"],
    "IVA": ["303","390"],
    "Sociedades": ["200","202"],
    "Patrimonio": ["714"],
    "Modelos-Informativas": ["720","721"],
    "GrandesEmpresas_SII": ["SII"],
    "Facturacion": [],
    "Discapacidad": [],
    "Aduanas_Transito": [],
}

DOC_HINTS = {
    "IRPF": [
        "Manual práctico de Renta 2024 (Parte 1), p. X",
        "Manual práctico de Renta 2024 (Parte 2), p. X"
    ],
    "IVA": [
        "Manual práctico IVA 2024, p. X",
        "Manual de facturación 2011, p. X"
    ],
    "Sociedades": [
        "Manual práctico de Sociedades 2024, p. X",
        "Manual Grandes Empresas, p. X"
    ],
    "Patrimonio": [
        "Manual práctico de Patrimonio 2024, p. X",
        "Manual Patrimonio 2024, p. X"
    ],
    "Modelos-Informativas": [
        "Manual práctico de Renta 2024 (informativas), p. X",
        "Manual práctico de Patrimonio 2024 (informativas), p. X"
    ],
    "Discapacidad": [
        "Manual específico para personas con discapacidad, p. X",
        "Manual práctico de Renta 2024 (Mínimos), p. X"
    ],
    "GrandesEmpresas_SII": [
        "Manual Grandes Empresas (SII), p. X",
        "Manual práctico IVA 2024 (Libros registro SII), p. X"
    ],
    "Aduanas_Transito": [
        "Manual de Tránsito, p. X",
        "Manual Grandes Empresas (Aduanas), p. X"
    ],
    "Facturacion": [
        "Manual de facturación 2011, p. X",
        "Manual práctico IVA 2024 (facturación), p. X"
    ],
    "Guardrails": [
        "Manual práctico de Renta 2024 (obligaciones), p. X",
        "Manual práctico IVA 2024 (obligaciones), p. X"
    ]
}

def guess_verdict(cat, q):
    if cat == "Guardrails":
        return "No — no puedo ayudarte a evadir impuestos."
    # muchas consultas requieren datos -> Depende
    for kw in ["¿", "cómo", "cuándo", "cuál", "puedo", "obligado"]:
        if kw in q.lower():
            return "Depende — faltan datos clave (año, CCAA, importes, régimen)."
    return random.choice([
        "Depende — faltan datos clave (año, CCAA, importes, régimen).",
        "Sí — con matices según CCAA y situación."
    ])

def bullets_why(cat):
    items = [
        "Las reglas pueden variar por comunidad autónoma.",
        "Pueden existir límites, mínimos exentos o bonificaciones.",
        "Es clave comprobar importes, fechas y régimen aplicable.",
        "La normativa y manuales establecen condiciones y excepciones."
    ]
    random.shuffle(items)
    return "\n".join(f"- {x}" for x in items[:3])

def missing_info(cat):
    items = [
        "Año fiscal y período de devengo.",
        "Comunidad autónoma de residencia y si hay normativa autonómica aplicable.",
        "Régimen (general/simplificado/estimar directa/atribución de rentas, etc.).",
        "Importes concretos (ingresos, gastos, patrimonio neto).",
        "Documentación justificativa (contratos, facturas, certificados)."
    ]
    random.shuffle(items)
    return "\n".join(f"- {x}" for x in items[:3])

def example_snip(cat):
    if cat == "Guardrails":
        return "No aplica."
    return (
        "Si el patrimonio neto supera X€ y no hay bonificación autonómica suficiente, "
        "la obligación de presentar podría activarse. Por ejemplo, con 2,3M€ y mínimo exento Y€, "
        "la base podría ser ~2,3M€-Y€ (aprox.), según manual."
    )

def models_for(cat):
    m = MODEL_HINTS.get(cat, [])
    return ", ".join(f"Modelo {x}" if x.isdigit() else x for x in m) if m else "No aparecen modelos en el contexto."

def citations_for(cat):
    docs = DOC_HINTS.get(cat, [])
    if len(docs) < 2:
        docs += ["Manual correspondiente, p. X"] * (2 - len(docs))
    return "\n".join(f"- {d}" for d in docs[:2])

def make_example(cat, q):
    # system
    system = FRIENDLY_SYSTEM
    # user
    user = q
    # assistant (solo estilo/estructura; contenido neutral)
    veredicto = guess_verdict(cat, q)
    content = FMT.format(
        veredicto=veredicto,
        resumen="Te explico de forma sencilla qué mirar y qué datos faltan para darte una respuesta cerrada.",
        bullets=bullets_why(cat),
        modelos=models_for(cat),
        faltantes=missing_info(cat),
        ejemplo=example_snip(cat),
        citas=citations_for(cat),
    )
    return {"messages":[
        {"role":"system","content":system},
        {"role":"user","content":user},
        {"role":"assistant","content":content}
    ]}

def make_refusal_example(q):
    system = FRIENDLY_SYSTEM
    user = q
    content = (
        "**Veredicto corto:** No — no puedo ayudar a evadir impuestos.\n"
        "**Resumen entendible:** No presto asistencia para ocultar ingresos o incumplir obligaciones. "
        "Sí puedo ayudarte a entender cómo regularizar y cumplir correctamente.\n"
        "**Por qué:**\n"
        "- Debes cumplir la normativa tributaria vigente.\n"
        "- El incumplimiento tiene sanciones y recargos.\n"
        "- Puedo orientarte sobre declaraciones complementarias o rectificativas.\n"
        "**Modelos/Formularios (si aplica):** Indeterminados; según el caso podrían ser complementarias o rectificativas.\n"
        "**Qué debes comprobar o aportar:**\n"
        "- Periodos afectados e importes.\n"
        "- Documentación soporte (facturas, extractos, nóminas).\n"
        "- Situación por impuesto (IVA/IRPF/IS) para ver el trámite correcto.\n"
        "**Ejemplo rápido (opcional):** Presentar declaración complementaria del periodo afectado con los importes omitidos.\n"
        "**Citas (mín. 2):**\n"
        "- Manual práctico de Renta 2024 (obligaciones), p. X\n"
        "- Manual práctico IVA 2024 (rectificativas), p. X\n"
        "**Aviso:** Esto no constituye asesoramiento profesional. Verifícalo con tu asesor."
    )
    return {"messages":[
        {"role":"system","content":system},
        {"role":"user","content":user},
        {"role":"assistant","content":content}
    ]}

# --- Construcción del dataset ---
dataset = []
for cat, qs in SEEDS.items():
    for q in qs:
        if cat == "Guardrails":
            dataset.append(make_refusal_example(q))
        else:
            dataset.append(make_example(cat, q))

# Mezclar y guardar
random.shuffle(dataset)
with open("ft_dataset.jsonl", "w", encoding="utf-8") as f:
    for ex in dataset:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✅ ft_dataset.jsonl generado con {len(dataset)} ejemplos.")

# --- Linter rápido: comprueba que las secciones estén presentes ---
def lint_example(msgs):
    assistant = next(m for m in msgs if m["role"]=="assistant")["content"]
    checks = {
        "veredicto": "**Veredicto corto:**" in assistant,
        "resumen": "**Resumen entendible:**" in assistant,
        "por_que": "**Por qué:**" in assistant,
        "modelos": "**Modelos/Formularios (si aplica):**" in assistant,
        "faltantes": "**Qué debes comprobar o aportar:**" in assistant,
        "citas": "**Citas (mín. 2):**" in assistant,
        "aviso": "**Aviso:**" in assistant,
    }
    return checks

bad = []
for i,ex in enumerate(dataset[:10]):  # muestreo
    ch = lint_example(ex["messages"])
    if not all(ch.values()):
        bad.append((i,ch))
print("Lint de muestra OK" if not bad else ("Problemas en:", bad[:3]))

✅ ft_dataset.jsonl generado con 26 ejemplos.
Lint de muestra OK


# === 4B) Ampliar dataset de FT con paráfrasis y variaciones (≈150–200 ejemplos) ===

In [10]:
import json, random, re

random.seed(123)

FRIENDLY_SYSTEM = (
    "Eres un asistente fiscal para ciudadanos y pymes en España. "
    "Respondes en español claro, directo y sin jerga; primero un veredicto (Sí/No/Depende), "
    "luego explicación sencilla. Incluyes ejemplos numéricos cuando ayudan. "
    "Siempre devuelves citaciones (doc y página) y un aviso legal. "
    "Si faltan datos (año, CCAA, régimen, importes), los pides. "
    "Rechazas peticiones ilegales (evadir impuestos) y propones vías de regularización. "
    "No inventes artículos ni modelos; el FT es de estilo."
)

BASE_SEEDS = [
    # IRPF
    ("IRPF", "¿Tengo que presentar la Renta si cobré 2 pagadores por 23.000€?"),
    ("IRPF", "¿Qué gastos puedo deducir por alquiler en IRPF 2024?"),
    ("IRPF", "¿Cómo tributan las criptomonedas en IRPF si hago staking?"),
    ("IRPF", "¿Puedo aplicar la deducción por maternidad este año?"),
    # IVA
    ("IVA", "Cómo rectificar el modelo 303 si declaré mal el IVA del último trimestre"),
    ("IVA", "¿Estoy obligado a presentar el 390 si hice 303 trimestrales en 2024?"),
    ("IVA", "¿Cómo funciona la inversión del sujeto pasivo en una obra?"),
    ("IVA", "¿Cuándo aplicar el tipo reducido del 10% en hostelería?"),
    # Sociedades
    ("Sociedades", "En el Impuesto sobre Sociedades, ¿qué gastos son no deducibles típicos?"),
    ("Sociedades", "¿Cómo compenso BINs en el modelo 200?"),
    ("Sociedades", "¿Cuándo tengo que presentar el modelo 202 de pagos fraccionados?"),
    # Patrimonio
    ("Patrimonio", "¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?"),
    ("Patrimonio", "¿Cómo se valora una participación no cotizada para Patrimonio?"),
    ("Patrimonio", "¿Qué mínimo exento aplica y cómo afecta la vivienda habitual?"),
    # Modelos informativas
    ("Informativas", "¿Cuándo se presenta el modelo 721 de cripto en el extranjero?"),
    ("Informativas", "¿Tengo que presentar el 720 por cuentas fuera de España?"),
    # Discapacidad
    ("Discapacidad", "¿Qué beneficios fiscales hay por discapacidad del 33% en IRPF?"),
    ("Discapacidad", "¿Cómo afecta la discapacidad a los mínimos familiares?"),
    # SII / Grandes Empresas
    ("SII", "Soy gran empresa, ¿qué plazos tengo en el SII?"),
    ("SII", "¿Cómo rectifico un asiento en el SII?"),
    # Aduanas / Tránsito
    ("Aduanas", "¿Cuáles son los documentos básicos para un régimen de tránsito?"),
    ("Aduanas", "¿Diferencias entre importación definitiva y tránsito?"),
    # Facturación
    ("Facturacion", "¿Qué datos obligatorios debe incluir una factura?"),
    ("Facturacion", "¿Cómo emitir una factura rectificativa correctamente?"),
    # Guardrails
    ("Guardrails", "¿Cómo puedo pagar menos impuestos ocultando ingresos?"),
    ("Guardrails", "¿Me dices cómo no declarar el IVA de unas operaciones?")
]

CCAA = ["Madrid","Cataluña","Andalucía","Valencia","Galicia","País Vasco","Navarra","Aragón","Castilla y León","Murcia"]
YEARS = ["2023","2024","2025"]
IMP_SAMPLES = ["18.500€","22.000€","30.000€","45.000€","2,3M€","3,1M€"]
PARAPHR = [
    "¿Me puedes decir si {q}?",
    "Tengo esta duda: {q}",
    "Consulta rápida: {q}",
    "En mi caso, {q}",
    "{q} ¿cómo lo haría?",
]

MODEL_HINTS = {
    "IRPF": ["100"],
    "IVA": ["303","390"],
    "Sociedades": ["200","202"],
    "Patrimonio": ["714"],
    "Informativas": ["720","721"],
    "SII": ["SII"],
    "Facturacion": [],
    "Discapacidad": [],
    "Aduanas": [],
}

DOC_HINTS = {
    "IRPF": [
        "Manual práctico de Renta 2024 (Parte 1), p. X",
        "Manual práctico de Renta 2024 (Parte 2), p. X"
    ],
    "IVA": [
        "Manual práctico IVA 2024, p. X",
        "Manual de facturación 2011, p. X"
    ],
    "Sociedades": [
        "Manual práctico de Sociedades 2024, p. X",
        "Manual Grandes Empresas, p. X"
    ],
    "Patrimonio": [
        "Manual práctico de Patrimonio 2024, p. X",
        "Manual Patrimonio 2024, p. X"
    ],
    "Informativas": [
        "Manual práctico de Renta 2024 (informativas), p. X",
        "Manual Patrimonio 2024 (informativas), p. X"
    ],
    "Discapacidad": [
        "Manual específico para personas con discapacidad, p. X",
        "Manual Renta 2024 (Mínimos), p. X"
    ],
    "SII": [
        "Manual Grandes Empresas (SII), p. X",
        "Manual práctico IVA 2024 (Libros SII), p. X"
    ],
    "Aduanas": [
        "Manual de Tránsito, p. X",
        "Manual Grandes Empresas (Aduanas), p. X"
    ],
    "Facturacion": [
        "Manual de facturación 2011, p. X",
        "Manual práctico IVA 2024 (facturación), p. X"
    ],
    "Guardrails": [
        "Manual práctico de Renta 2024 (obligaciones), p. X",
        "Manual práctico IVA 2024 (obligaciones), p. X"
    ]
}

FMT = (
"**Veredicto corto:** {veredicto}\n"
"**Resumen entendible:** {resumen}\n"
"**Por qué:**\n{bullets}\n"
"**Modelos/Formularios (si aplica):** {modelos}\n"
"**Qué debes comprobar o aportar:**\n{faltantes}\n"
"**Ejemplo rápido (opcional):** {ejemplo}\n"
"**Citas (mín. 2):**\n{citas}\n"
"**Aviso:** Esto no constituye asesoramiento profesional. Verifícalo con tu asesor."
)

def paraphrase(q):
    q = q.replace("2024", random.choice(YEARS))
    q = q.replace("Madrid", random.choice(CCAA))
    q = re.sub(r"\d{1,2}[.,]?\d{3}€|\d+(?:,\d+)?M€", random.choice(IMP_SAMPLES), q)
    tpl = random.choice(PARAPHR)
    return tpl.format(q=q)

def verdict(cat, q):
    if cat == "Guardrails":
        return "No — no puedo ayudarte a evadir impuestos."
    return random.choice([
        "Depende — faltan datos (año, CCAA, importes, régimen).",
        "Sí — con condiciones según CCAA y situación.",
        "Depende — revisa mínimos exentos, límites y bonificaciones."
    ])

def bullets(cat):
    base = [
        "Puede variar por comunidad autónoma (normativa autonómica).",
        "Comprueba límites, mínimos exentos o bonificaciones aplicables.",
        "Importa el régimen y las fechas/periodos.",
        "La documentación soporte (facturas, contratos, certificados) es clave."
    ]
    random.shuffle(base)
    return "\n".join(f"- {x}" for x in base[:3])

def missing(cat):
    ls = [
        "Año fiscal y periodo de devengo.",
        "Comunidad autónoma de residencia.",
        "Régimen aplicable (general, simplificado, estimación, etc.).",
        "Importes concretos (ingresos, gastos, patrimonio neto)."
    ]
    random.shuffle(ls)
    return "\n".join(f"- {x}" for x in ls[:3])

def example(cat):
    if cat == "Guardrails":
        return "No aplica."
    return "Ejemplo orientativo con importes redondeados para entender los pasos, sin sustituir el cálculo oficial."

def models(cat):
    xs = MODEL_HINTS.get(cat, [])
    return ", ".join(f"Modelo {m}" if m.isdigit() else m for m in xs) if xs else "No aparecen modelos en el contexto."

def citations(cat):
    cs = DOC_HINTS.get(cat, [])
    return "\n".join(f"- {c}" for c in cs[:2])

def make_item(cat, q):
    content = FMT.format(
        veredicto=verdict(cat,q),
        resumen="Te explico qué mirar y qué datos faltan para dar una respuesta cerrada.",
        bullets=bullets(cat),
        modelos=models(cat),
        faltantes=missing(cat),
        ejemplo=example(cat),
        citas=citations(cat),
    )
    return {
        "messages": [
            {"role":"system","content":FRIENDLY_SYSTEM},
            {"role":"user","content":paraphrase(q)},
            {"role":"assistant","content":content}
        ]
    }

dataset = []
for cat, q in BASE_SEEDS:
    # más variaciones por semilla
    n = 6 if cat not in ("Guardrails","Aduanas","Facturacion") else 3
    for _ in range(n):
        dataset.append(make_item(cat, q))

random.shuffle(dataset)
with open("ft_dataset.jsonl","a",encoding="utf-8") as f:  # append al existente
    for ex in dataset:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("✅ Añadidos", len(dataset), "ejemplos. Total (aprox) ahora ~150–200 en ft_dataset.jsonl")


✅ Añadidos 138 ejemplos. Total (aprox) ahora ~150–200 en ft_dataset.jsonl


# === 5) Validador de formato FRIENDLY y citaciones mínimas ===

In [11]:

import re

SECTION_REGEX = {
    "veredicto": r"\*\*Veredicto corto:\*\*",
    "resumen": r"\*\*Resumen entendible:\*\*",
    "porque": r"\*\*Por qué:\*\*",
    "modelos": r"\*\*Modelos/Formularios \(si aplica\):\*\*",
    "faltantes": r"\*\*Qué debes comprobar o aportar:\*\*",
    "citas": r"\*\*Citas \(mín\. 2\):\*\*",
    "aviso": r"\*\*Aviso:\*\*",
}

def eval_friendly(txt: str) -> dict:
    checks = {k: bool(re.search(rx, txt, flags=re.I)) for k,rx in SECTION_REGEX.items()}

    # Contar bullets de citas y presencia de p.X
    citas_block = re.split(SECTION_REGEX["citas"], txt, flags=re.I)
    cita_lines = []
    if len(citas_block) > 1:
        after = citas_block[1]
        # hasta el siguiente encabezado ** o final
        after = re.split(r"\n\*\*", after)[0]
        cita_lines = [ln for ln in after.splitlines() if ln.strip().startswith("- ")]
    checks["citas_min_2"] = len(cita_lines) >= 2
    checks["citaciones_con_p"] = any(re.search(r"\bp\.\s*\d+", ln) for ln in cita_lines)

    # Aviso legal explícito
    checks["aviso_texto"] = "no constituye asesoramiento profesional" in txt.lower()

    return checks

# Prueba rápida
demo = """**Veredicto corto:** Depende...
**Resumen entendible:** ...
**Por qué:**
- 1
- 2
**Modelos/Formularios (si aplica):** Modelo 714
**Qué debes comprobar o aportar:**
- año
- ccaa
**Ejemplo rápido (opcional):** ...
**Citas (mín. 2):**
- Manual práctico de Patrimonio 2024, p. 17
- Manual Patrimonio 2024, p. 70
**Aviso:** Esto no constituye asesoramiento profesional. Verifícalo con tu asesor."""
print(eval_friendly(demo))


{'veredicto': True, 'resumen': True, 'porque': True, 'modelos': True, 'faltantes': True, 'citas': True, 'aviso': True, 'citas_min_2': True, 'citaciones_con_p': True, 'aviso_texto': True}


# === 5B) Probar la validación de modelos permitidos con una respuesta real ===

In [13]:

# 1) Lanza una pregunta y genera respuesta friendly con RAG
q = "¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?"
answer = generate_answer_openai_friendly(q, k=6)

# 2) Obtén los modelos permitidos por el CONTEXTO (no inventados)
rows = retrieve(q, k=6)
allowed_models = extract_models_from_rows(rows)  # viene de la 3B-FRIENDLY+

print("=== Modelos detectados en contexto ===")
print(allowed_models if allowed_models else "ninguno")
print()

# 3) Valida que los modelos citados en la respuesta estén dentro de los permitidos
check_models = eval_models_allowed(answer, allowed_models)
print("=== Resultado validación modelos ===")
print(check_models)  # {'cited': [...], 'allowed': [...], 'ok': True/False}
print()

# 4) (Opcional) Mira la respuesta producida
print("=== Respuesta ===")
print(answer)

===== Pasajes recuperados =====
- Manual_práctico_de_Patrimonio_2024..pdf  p.17
- Manual_Patrimonio_2024.pdf  p.70
- Manual_Patrimonio_2024.pdf  p.13
- Manual_práctico_de_Patrimonio_2024..pdf  p.7
- Manual_práctico_de_Patrimonio_2024..pdf  p.91
- Manual_práctico_de_Patrimonio_2024..pdf  p.6
===== Modelos detectados en contexto =====
ninguno

=== Modelos detectados en contexto ===
ninguno

=== Resultado validación modelos ===
{'cited': [], 'allowed': [], 'ok': True}

=== Respuesta ===
**Veredicto corto:** Sí, estás obligado a presentar la declaración.  
**Resumen entendible:** Si tu patrimonio supera los 2.000.000 de euros, como en tu caso con 2,3M€, debes presentar la declaración del Impuesto sobre el Patrimonio en Madrid.  
**Por qué:**  
- La obligación de presentar se da si el valor de tus bienes o derechos es superior a 2.000.000 euros.  
- Esto se aplica independientemente de si la cuota tributaria resulta a ingresar o no.  
- La normativa es clara en establecer este umbral para l

# === 5C) Validación completa de una respuesta (formato FRIENDLY + citaciones + modelos) ===

In [14]:

def evaluate_answer(question: str, k: int = 6) -> dict:
    # Genera respuesta
    answer = generate_answer_openai_friendly(question, k=k)
    # Formato/citaciones (usa eval_friendly de la celda 5)
    fmt = eval_friendly(answer)
    # Modelos permitidos por el CONTEXTO
    rows = retrieve(question, k=k)
    allowed = extract_models_from_rows(rows)
    mdl = eval_models_allowed(answer, allowed)
    # Aprobado si todas las secciones están y los modelos son coherentes
    ok = all(fmt.values()) and mdl["ok"]
    return {
        "ok": ok,
        "format_checks": fmt,
        "model_checks": mdl,
        "allowed_models": allowed,
        "answer": answer
    }

# 🔎 Ejemplo de uso:
res = evaluate_answer("¿Estoy obligado a presentar Patrimonio 2024 en Madrid con 2,3M€?")
print("OK global:", res["ok"])
print("\n-- Checks de formato --")
print(res["format_checks"])
print("\n-- Checks de modelos --")
print(res["model_checks"])
print("\n-- Modelos permitidos por contexto --")
print(res["allowed_models"])
print("\n-- Respuesta --\n")
print(res["answer"])


===== Pasajes recuperados =====
- Manual_práctico_de_Patrimonio_2024..pdf  p.17
- Manual_Patrimonio_2024.pdf  p.70
- Manual_Patrimonio_2024.pdf  p.13
- Manual_práctico_de_Patrimonio_2024..pdf  p.7
- Manual_práctico_de_Patrimonio_2024..pdf  p.91
- Manual_práctico_de_Patrimonio_2024..pdf  p.6
===== Modelos detectados en contexto =====
ninguno

OK global: False

-- Checks de formato --
{'veredicto': True, 'resumen': True, 'porque': True, 'modelos': True, 'faltantes': True, 'citas': False, 'aviso': True, 'citas_min_2': False, 'citaciones_con_p': False, 'aviso_texto': True}

-- Checks de modelos --
{'cited': [], 'allowed': [], 'ok': True}

-- Modelos permitidos por contexto --
[]

-- Respuesta --

**Veredicto corto:** Sí, estás obligado a presentar la declaración.

**Resumen entendible:** Si tu patrimonio supera los 2.000.000 de euros, como en tu caso con 2,3 millones, debes presentar la declaración del Impuesto sobre el Patrimonio en Madrid.

**Por qué:**
- La obligación de presentar la de